In [1]:
import pandas as pd
import json

# Load ZEOSYN.csv
zeosyn = pd.read_csv('ZEOSYN.csv')

# Load clean_code1 file - this is a dict mapping framework codes to lists of DOIs
with open('outputs/clean_code1_doi_mapping_20251112_095305.json', 'r') as f:
    clean_code1_dict = json.load(f)

# Convert to a long-form DataFrame with framework_code and doi columns
data_rows = []
for framework_code, doi_list in clean_code1_dict.items():
    for doi in doi_list:
        data_rows.append({'framework_code': framework_code, 'doi': doi})

clean_code1 = pd.DataFrame(data_rows)

# Show header columns and first few lines
print(f"Total entries: {len(clean_code1)}")
print(f"Unique framework codes: {clean_code1['framework_code'].nunique()}")
print(f"Unique DOIs: {clean_code1['doi'].nunique()}")
print("\nColumns:", clean_code1.columns.tolist())
print("\nFirst 10 rows:")
print(clean_code1.head(10))

Total entries: 544
Unique framework codes: 113
Unique DOIs: 380

Columns: ['framework_code', 'doi']

First 10 rows:
  framework_code                                 doi
0            SOS              10.1002/anie.200461911
1            JRY              10.1002/anie.200803578
2            POS              10.1002/anie.201309766
3           NUD1              10.1002/anie.201404608
4            LTA              10.1002/anie.201404608
5            LTA     10.1016/j.micromeso.2007.06.019
6            LTA       10.1021/acs.chemmater.5b04439
7            LTA        10.1007/978-3-662-47395-5_13
8            LTA        10.1016/j.carbon.2005.08.010
9            LTA  10.1016/j.materresbull.2010.05.033


/tmp/ipykernel_333147/3056108378.py:5: DtypeWarning: Columns (63,73) have mixed types. Specify dtype option on import or set low_memory=False.
  zeosyn = pd.read_csv('ZEOSYN.csv')


In [2]:
import re

# Define the columns we want to extract
columns_to_extract = [
    'osda1', 'osda2', 'osda3', 
    'product1', 'product2', 'product3', 
    'precursors',
    'osda1 synonyms', 'osda2 synonyms', 'osda3 synonyms',
    'osda1 iupac', 'osda2 iupac', 'osda3 iupac',
    'osda1 smiles', 'osda2 smiles', 'osda3 smiles',
    'osda1 formula', 'osda2 formula', 'osda3 formula',
    'Code1', 'Code2', 'Code3'
]

# Normalize DOI function (same as previous notebook)
def normalize_doi(s):
    if pd.isna(s) or s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r'^(https?://(dx\.)?doi\.org/)|^doi:\s*', '', s, flags=re.I)
    return s.strip().lower()

# Create normalized DOI column in zeosyn
zeosyn['doi_normalized'] = zeosyn['doi'].apply(normalize_doi)

# Normalize DOIs in clean_code1
clean_code1['doi_normalized'] = clean_code1['doi'].apply(normalize_doi)

print(f"ZEOSYN rows with valid DOIs: {(zeosyn['doi_normalized'] != '').sum()}")
print(f"clean_code1 rows: {len(clean_code1)}")
print(f"Unique normalized DOIs in clean_code1: {clean_code1['doi_normalized'].nunique()}")

ZEOSYN rows with valid DOIs: 23961
clean_code1 rows: 544
Unique normalized DOIs in clean_code1: 380


In [3]:
# Build framework to text mapping
framework_texts = {}
frameworks_no_match = []

for framework_code in clean_code1_dict.keys():
    # Get all DOIs for this framework
    framework_dois = clean_code1[clean_code1['framework_code'] == framework_code]['doi_normalized'].tolist()
    
    # Find all matching rows in zeosyn
    matching_rows = zeosyn[zeosyn['doi_normalized'].isin(framework_dois)]
    
    if len(matching_rows) == 0:
        frameworks_no_match.append(framework_code)
        continue
    
    # Extract and concatenate text from specified columns
    text_parts = []
    for idx, row in matching_rows.iterrows():
        row_text_parts = []
        for col in columns_to_extract:
            value = row[col]
            # Only include if not null/empty
            if pd.notna(value) and str(value).strip() != '':
                row_text_parts.append(f"{col}: {value}")
        
        if row_text_parts:  # Only add if there's actual content
            text_parts.append(" | ".join(row_text_parts))
    
    # Join all rows with newline delimiter
    if text_parts:
        framework_texts[framework_code] = "\n".join(text_parts)

# Create output DataFrame
output_df = pd.DataFrame([
    {'framework': fw, 'text': text} 
    for fw, text in framework_texts.items()
])

# Report results
print(f"Total frameworks in clean_code1: {len(clean_code1_dict)}")
print(f"Frameworks with matching ZEOSYN data: {len(framework_texts)}")
print(f"Frameworks with NO matches: {len(frameworks_no_match)}")
if frameworks_no_match:
    print(f"\nFrameworks without matches: {frameworks_no_match}")

print(f"\nOutput DataFrame shape: {output_df.shape}")
print(f"\nFirst framework example:")
print(f"Framework: {output_df.iloc[0]['framework']}")
print(f"Text preview (first 500 chars):\n{output_df.iloc[0]['text'][:500]}...")

Total frameworks in clean_code1: 113
Frameworks with matching ZEOSYN data: 113
Frameworks with NO matches: 0

Output DataFrame shape: (113, 2)

First framework example:
Framework: SOS
Text preview (first 500 chars):
osda1: diethylenetriamine | osda2: pyridine | product1: SU-16 | precursors: GeO2, H3BO3 | osda1 synonyms: ['N-(2-aminoethyl)ethane-1,2-diamine', 'Bis(2-aminoethyl)amine', '111-40-0', '73989-30-7', '8076-55-9', '94700-17-1', '98824-35-2', '59135-90-9', '26915-78-6', '53303-76-7', '54018-92-7', '(Aminoethyl)ethanediamine', '1,2-Ethanediamine, N-(2-aminoethyl)-', "2,2'-Diaminodiethylamine", '3-Azapentane-1,5-diamine', 'Barsamide 115', 'Bis(2-aminoethyl)amine', 'Bis[.beta.-aminoethyl]amine', "Diethy...


In [4]:
# Save to CSV
output_path = 'framework_text_mapping.csv'
output_df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

# Show some statistics about text lengths
output_df['text_length'] = output_df['text'].str.len()
print(f"\nText length statistics:")
print(output_df['text_length'].describe())


Saved to framework_text_mapping.csv

Text length statistics:
count       113.000000
mean      34503.274336
std       73010.325980
min          81.000000
25%        2597.000000
50%       11561.000000
75%       31679.000000
max      569866.000000
Name: text_length, dtype: float64


In [5]:
# Display a couple of example framework text strings
print("Example 1:")
print(f"Framework: {output_df.iloc[0]['framework']}")
print(f"Text:\n{output_df.iloc[0]['text']}\n")
print("="*80)
print("\nExample 2:")
print(f"Framework: {output_df.iloc[5]['framework']}")
print(f"Text:\n{output_df.iloc[5]['text']}\n")

Example 1:
Framework: SOS
Text:
osda1: diethylenetriamine | osda2: pyridine | product1: SU-16 | precursors: GeO2, H3BO3 | osda1 synonyms: ['N-(2-aminoethyl)ethane-1,2-diamine', 'Bis(2-aminoethyl)amine', '111-40-0', '73989-30-7', '8076-55-9', '94700-17-1', '98824-35-2', '59135-90-9', '26915-78-6', '53303-76-7', '54018-92-7', '(Aminoethyl)ethanediamine', '1,2-Ethanediamine, N-(2-aminoethyl)-', "2,2'-Diaminodiethylamine", '3-Azapentane-1,5-diamine', 'Barsamide 115', 'Bis(2-aminoethyl)amine', 'Bis[.beta.-aminoethyl]amine', "Diethylamine, 2,2'-diamino-", 'Diethylenetriamine', "Ethylamine, 2,2'-iminobis-", 'Ethylenediamine, N-(2-aminoethyl)-', 'N,N-Bis(2-aminoethyl)amine', 'NSC446', 'WLN: Z2M2Z', '1,4,7-Triazaheptane', '1,5-Diamino-3-azapentane', "2,2'-Iminobis(ethanamine)", "2,2'-Iminodi(ethylamine)", '2-(2-Aminoethylamino)ethylamine', '4-04-00-01238 (Beilstein Handbook Reference)', 'Aminoethylethandiamine', 'Ancamine DETA', 'BRN 0605314', 'Bis(beta-aminoethyl)amine', 'CCRIS 4794', 'ChS-P 1

In [6]:
import ast

# Initialize list to store framework records
framework_records = []

# Get unique framework codes from clean_code1_dict
for framework_code in clean_code1_dict.keys():
    # Get all DOIs for this framework
    framework_dois = clean_code1[clean_code1['framework_code'] == framework_code]['doi_normalized'].tolist()
    
    # Find all matching rows in zeosyn
    matching_rows = zeosyn[zeosyn['doi_normalized'].isin(framework_dois)]
    
    if len(matching_rows) == 0:
        continue
    
    # Collect cryst_temp values
    cryst_temps = []
    for val in matching_rows['cryst_temp'].dropna():
        if str(val).strip() != '':
            cryst_temps.append(str(val).strip())
    
    # Collect cryst_time values
    cryst_times = []
    for val in matching_rows['cryst_time'].dropna():
        if str(val).strip() != '':
            cryst_times.append(str(val).strip())
    
    # Collect OSDA values from multiple columns
    osda_values = []
    osda_columns = ['osda1', 'osda2', 'osda3', 'osda1 synonyms', 'osda2 synonyms', 'osda3 synonyms']
    for idx, row in matching_rows.iterrows():
        for col in osda_columns:
            val = row[col]
            if pd.notna(val) and str(val).strip() != '':
                # Check if it's a synonyms column (contains list-like string)
                if 'synonyms' in col:
                    try:
                        # Try to parse as Python list
                        synonym_list = ast.literal_eval(str(val))
                        if isinstance(synonym_list, list):
                            osda_values.extend([str(s).strip() for s in synonym_list if str(s).strip() != ''])
                        else:
                            osda_values.append(str(val).strip())
                    except:
                        # If parsing fails, treat as single value
                        osda_values.append(str(val).strip())
                else:
                    osda_values.append(str(val).strip())
    
    # Remove duplicates while preserving order
    osda_values = list(dict.fromkeys(osda_values))
    
    # Collect precursor values
    precursors = []
    for val in matching_rows['precursors'].dropna():
        if str(val).strip() != '':
            precursors.append(str(val).strip())
    
    # Create record
    record = {
        'framework_code': framework_code,
        'cryst_temp': ', '.join(cryst_temps) if cryst_temps else '',
        'cryst_time': ', '.join(cryst_times) if cryst_times else '',
        'OSDA': ', '.join(osda_values) if osda_values else '',
        'precursor': ', '.join(precursors) if precursors else ''
    }
    
    framework_records.append(record)

# Save to JSON
output_json_path = 'framework_synthesis_data.json'
with open(output_json_path, 'w') as f:
    json.dump(framework_records, f, indent=2)

print(f"Created JSON file with {len(framework_records)} frameworks")
print(f"Saved to {output_json_path}\n")

# Print first few rows for verification
print("First 3 records:")
for i, record in enumerate(framework_records[:3]):
    print(f"\n--- Record {i+1} ---")
    for key, value in record.items():
        # Truncate long values for display
        display_value = value if len(str(value)) <= 200 else str(value)[:200] + "..."
        print(f"{key}: {display_value}")

Created JSON file with 113 frameworks
Saved to framework_synthesis_data.json

First 3 records:

--- Record 1 ---
framework_code: SOS
cryst_temp: 165.0
cryst_time: 192.0
OSDA: diethylenetriamine, pyridine, N-(2-aminoethyl)ethane-1,2-diamine, Bis(2-aminoethyl)amine, 111-40-0, 73989-30-7, 8076-55-9, 94700-17-1, 98824-35-2, 59135-90-9, 26915-78-6, 53303-76-7, 54018-92-7, (Amin...
precursor: GeO2, H3BO3

--- Record 2 ---
framework_code: JRY
cryst_temp: 180.0, 180.0
cryst_time: 288.0, 288.0
OSDA: diethylamine, N-ethylethanamine, DIETHYLAMINE, 109-89-7, D0806_SIAL, NCGC00090709-01, 386456_ALDRICH, ST5214507, 31730_FLUKA, InChI=1/C4H11N/c1-3-5-4-2/h5H,3-4H2,1-2H, AI3-24215, CCRIS 4792, Diaethyla...
precursor: H3PO4, tetraethylene glycol, Co(NO3)2*6H2O, Al(OiPr)3, H3PO4, tetraethylene glycol, Zn(NO3)2*9H20, Al(OiPr)3

--- Record 3 ---
framework_code: POS
cryst_temp: 125.0
cryst_time: 336.0
OSDA: 4-dimethylaminopyridine, N,N-Dimethylpyridin-4-amine, N,N-DIMETHYL-4-PYRIDINAMINE, dimethyl-(4-pyrid